# VOD archive: transcription on Colab

GPU stage of the pipeline (voice enrollment and `process.py`). Used instead of `deploy/gpu/run_remote.sh`.

**Once:**
- Runtime → Change runtime type → **T4 GPU** (or better).
- 🔑 Secrets (left sidebar): add `HF_TOKEN` (the Hugging Face token) and allow notebook access.

**Each run:**
1. On the laptop: `deploy/colab/pack.sh process --model large-v3` (includes `voices/`, for `TASK = "both"`), or `deploy/colab/pack.sh enroll`.
2. Upload `colab/bundle.tar` to Google Drive as **My Drive/vod-archive/bundle.tar** (replace the old one).
3. Set `TASK` / `ARGS` below, then Runtime → **Run all**.
4. Download **My Drive/vod-archive/out** from Drive, then on the laptop: `deploy/colab/unpack.sh <the download>`.

Transcripts are written straight to Drive (`vod-archive/out/transcripts`), so if Colab disconnects, run all
again: finished VODs are skipped.

Everything the notebook runs, and any error, is also written to **My Drive/vod-archive/out/colab.log**, so the
laptop (`rclone cat gdrive:out/colab.log`) can read what happened without copying from this page.

In [ ]:
TASK = "process"            # "enroll", "process", or "both" (enroll, then process with the new voiceprints)
ARGS = "--model large-v3"   # process.py arguments (pilot: run once with large-v3, once with large-v3-turbo)
DRIVE_DIR = "/content/drive/MyDrive/vod-archive"
REPO = "/content/thehasanproject"

In [ ]:
import os, subprocess, sys, time, traceback
from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = f"{DRIVE_DIR}/out"
os.makedirs(OUT, exist_ok=True)
LOG = f"{OUT}/colab.log"

def log(text):
    with open(LOG, "a") as f:
        f.write(text)

def run(cmd):
    """Run a shell command, show its output here and copy it to the log on Drive."""
    log(f"\n$ {cmd}\n")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(LOG, "a") as f:
        for line in p.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
    p.wait()
    log(f"[exit {p.returncode}]\n")
    if p.returncode:
        raise RuntimeError(f"failed (exit {p.returncode}): {cmd}  (full output in {LOG})")

def log_error(shell, etype, value, tb, tb_offset=None):
    log("\n!!! ERROR\n" + "".join(traceback.format_exception(etype, value, tb)))
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
get_ipython().set_custom_exc((Exception,), log_error)

log(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S %Z')}  TASK={TASK}  ARGS={ARGS}\n")
# strip: a token pasted with a trailing space or newline breaks every Hugging Face request
os.environ["HF_TOKEN"] = (userdata.get("HF_TOKEN") or "").strip()
assert os.environ["HF_TOKEN"], "add HF_TOKEN in Secrets (key icon) and allow notebook access"
assert os.path.exists(f"{DRIVE_DIR}/bundle.tar"), f"upload bundle.tar to {DRIVE_DIR} first"
run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
if os.path.isdir(REPO):
    run(f"git -C {REPO} pull -q")
else:
    run(f"git clone -q https://github.com/carrier3rd/thehasanproject {REPO}")
os.chdir(REPO)
run("git log -1 --format='code: %h %s'")
run("pip install -q -r requirements-gpu.txt")

In [ ]:
# inputs from the bundle; outputs go straight to Drive so a disconnect loses at most the VOD in progress
run(f"tar -xf {DRIVE_DIR}/bundle.tar -C {REPO}")
for sub in ("transcripts", "logs", "voiceprints"):
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
for link, target in (("data/transcripts", f"{OUT}/transcripts"), ("logs", f"{OUT}/logs")):
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        run(f"cp -rn {link}/. {target}/ && rm -rf {link}")
    os.makedirs(os.path.dirname(link) or ".", exist_ok=True)
    os.symlink(target, link)
os.environ["VOD_ARCHIVE_PRIVATE_DIR"] = "private"
run("python check_env.py --role gpu || true")
run("ls -l data/audio; ls -R voices 2>/dev/null || true")

In [ ]:
assert TASK in ("enroll", "process", "both"), "TASK must be 'enroll', 'process' or 'both'"
if TASK in ("enroll", "both"):
    run("python pipeline/enroll_voices.py")
    run(f"cp voiceprints/*.npy {OUT}/voiceprints/ && ls -l {OUT}/voiceprints")
if TASK in ("process", "both"):
    run(f"python pipeline/process.py {ARGS}")
    run(f"find {OUT}/transcripts -name '*.json' | sort")

In [ ]:
log(f"===== finished {time.strftime('%Y-%m-%d %H:%M:%S %Z')}\n")
drive.flush_and_unmount()   # make sure everything reached Drive before you download it
print("Done. Download My Drive/vod-archive/out, or let the laptop fetch it with rclone.")